In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchvision.models as models
import os, cv2, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


In [2]:
def corrupt_image(img):
    def add_noise(i):
        return np.clip(i+np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i):
        return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([add_noise,blur,lowres])(img)


class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]

        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        corrupted=corrupt_image(img.copy())

        return self.transform(corrupted), self.transform(img)


In [3]:
DATA_PATH = r"D:\Image Recognition\data"
dataset = RAImageDataset(DATA_PATH)
loader = DataLoader(dataset,batch_size=8,shuffle=True)


In [4]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,1,1),
            nn.BatchNorm2d(b),
            nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),
            nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [5]:
device="cuda" if torch.cuda.is_available() else "cpu"

teacher = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
teacher = teacher.to(device)
teacher.eval()

# classifier normalization
normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)
print(device)


cuda


In [6]:
model=ProcessingNet().to(device)
model.load_state_dict(torch.load("baseline_model.pth"))


<All keys matched successfully>

In [7]:
image_loss_fn = nn.MSELoss()
class_loss_fn = nn.CrossEntropyLoss()

lambda_recog = 0.1
optimizer = optim.Adam(model.parameters(),lr=1e-4)


In [8]:
for epoch in range(10):

    total_loss=0
    pbar=tqdm(loader)

    for corrupted, clean in pbar:

        corrupted=corrupted.to(device)
        clean=clean.to(device)

        # restore image
        restored=model(corrupted)

        # ----- Image Quality Loss -----
        img_loss=image_loss_fn(restored,clean)

        # ----- Recognition Loss -----
        with torch.no_grad():
            teacher_pred=teacher(normalize(clean))
            labels=teacher_pred.argmax(dim=1)

        pred=teacher(normalize(restored))
        recog_loss=class_loss_fn(pred,labels)

        # ----- Combined Loss -----
        loss=img_loss + lambda_recog*recog_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        pbar.set_postfix(total=loss.item(),img=img_loss.item(),cls=recog_loss.item())

    print("Epoch Loss:",total_loss/len(loader))


100%|██████████| 375/375 [02:43<00:00,  2.29it/s, cls=0.78, img=0.00173, total=0.0797]  


Epoch Loss: 0.09535216914614042


100%|██████████| 375/375 [02:36<00:00,  2.39it/s, cls=0.565, img=0.00178, total=0.0583] 


Epoch Loss: 0.08972564362486203


100%|██████████| 375/375 [02:35<00:00,  2.42it/s, cls=1.84, img=0.00158, total=0.185]   


...

Epoch Loss: 0.08024812315963209


100%|██████████| 375/375 [02:44<00:00,  2.28it/s, cls=0.766, img=0.00245, total=0.079]  


Epoch Loss: 0.08298844021558761


100%|██████████| 375/375 [02:41<00:00,  2.32it/s, cls=0.658, img=0.00207, total=0.0678]

Epoch Loss: 0.08046856071054935


In [9]:
torch.save(model.state_dict(),"RA_model.pth")
print("Recognition-Aware model saved as RA_model.pth")


Recognition-Aware model saved as RA_model.pth
